# MATS Portfolio Backtest — Colab Runner

> **Just hit `Ctrl+F9` (Run All) and walk away.**
>
> For details, troubleshooting, and lessons learned, see [`INSTALLME.Colab.md`](https://github.com/aistudylearning/mats-code/blob/main/INSTALLME.Colab.md)

| Cell | What it does | Time |
|---|---|---|
| 1 — Setup | Install deps, mount Drive, clone repo, copy data to SSD | ~15–20 min |
| 2 — Backtest | Run 50-asset × 10-timeframe portfolio backtest | ~2–3 hours |
| 3 — (Optional) | Manual report rescue if auto-copy failed | ~5 sec |

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║     Cell 1: MATS Colab Master Setup — run once per session      ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── Step 1: Install dependencies ────────────────────────────────────
# Note: pandas-ta manages its own pandas version. Do NOT pin pandas.
!pip install -q polars pyarrow duckdb pandas-ta ccxt joblib

# ── Step 2: Mount Google Drive ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Step 3: Clone or update the repo ────────────────────────────────
import os, shutil, time
if not os.path.exists('/content/mats-code'):
    !git clone https://github.com/aistudylearning/mats-code.git /content/mats-code
else:
    !cd /content/mats-code && git pull

# ── Step 4: Copy data from Drive → local SSD (~15 min, ~2× speedup)
# Drive I/O: ~50 MB/s   |   Colab local SSD: ~1 GB/s
DRIVE_DATA = "/content/drive/MyDrive/trading/raw/data/hot/data"
LOCAL_DATA  = "/content/data"

if not os.path.exists(LOCAL_DATA):
    print("⏳ Copying data to local SSD (one-time, ~15 min)...")
    t0 = time.time()
    shutil.copytree(DRIVE_DATA, LOCAL_DATA)
    print(f"✅ Data copied in {(time.time()-t0)/60:.1f} min")
else:
    print("✅ Local data already present — skipping copy")

# ── Step 5: Configure environment ───────────────────────────────────
os.environ["MATS_DATA_ROOT"] = LOCAL_DATA
%cd /content/mats-code

# ── Step 6: Verify ──────────────────────────────────────────────────
print("\n✅ MATS ready.")
print("   DATA_ROOT :", os.environ["MATS_DATA_ROOT"])
print("   First assets:", os.listdir(LOCAL_DATA)[:5])

---
## 🚀 Backtest

The cell below runs the full 50-asset × 10-timeframe portfolio backtest.

- **`-u`** = unbuffered output (real-time logging)
- **`tee`** = saves log to Drive AND shows it on screen
- **HTML report** auto-saves to `My Drive → trading → reports`

> ⏱ Expected runtime: ~2–3 hours on free tier

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║     Cell 2: Portfolio Backtest — 50 assets × 10 timeframes      ║
# ╚══════════════════════════════════════════════════════════════════╝

!python3 -u main.py portfolio \
    --signal 0.2 \
    --timeframe 1m 5m 15m 30m 1h 2h 4h 1D 1W 1M \
    --html \
    2>&1 | tee /content/drive/MyDrive/trading/reports/log_backtest_$(date +%Y%m%d_%H%M).txt

print("\n🏁 Backtest complete!")

---
## 🛟 Optional: Manual Report Rescue

Only run this if the HTML report did NOT appear in `My Drive → trading → reports`.

In [ ]:
# ── Only if auto-copy failed ────────────────────────────────────────
import shutil, glob, os
for f in glob.glob('/content/mats-code/output/*.html'):
    dest = f"/content/drive/MyDrive/trading/reports/{os.path.basename(f)}"
    shutil.copy2(f, dest)
    print("✅ Saved:", dest)